In [1]:
import os
import requests
import pandas as pd

from dotenv import load_dotenv

In [2]:
load_dotenv()

SEOUL_API_KEY = os.getenv("SEOUL_API_KEY")

In [3]:
SEOUL_API_KEY is not None

True

# 데이터 불러오기

## 테스트

In [4]:
url = (
    f"http://openapi.seoul.go.kr:8088/"
    f"{SEOUL_API_KEY}/json/culturalEventInfo/1/5/"
)

response = requests.get(url)

response.status_code

200

In [5]:
data = response.json()

data.keys()

dict_keys(['culturalEventInfo'])

In [6]:
data["culturalEventInfo"].keys()

dict_keys(['list_total_count', 'RESULT', 'row'])

In [7]:
rows = data["culturalEventInfo"]["row"]

In [8]:
df = pd.DataFrame(rows)

df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,클래식,마포구,[마포문화재단] 제11회 M 클래식 축제 [뮤라벨 콘서트],2026-10-28~2026-10-28,마포아트센터 플레이맥,마포문화재단,8세이상 관람가능,"전석 20,000원","02-3274-8600 [문의1번] 평일 9:00 - 18:00 (토,일 공휴일 휴무)",,...,2026-07-08,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,(수) 19:30


In [9]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

## 분석 데이터 불러오기

In [10]:
total_count = data["culturalEventInfo"]["list_total_count"]
all_rows = []

for start in range(1, total_count + 1, 1000):
    end = min(start + 999, total_count)

    url = (
        f"http://openapi.seoul.go.kr:8088/"
        f"{SEOUL_API_KEY}/json/culturalEventInfo/"
        f"{start}/{end}/"
    )

    response = requests.get(url)
    result = response.json()

    rows = result["culturalEventInfo"]["row"]
    all_rows.extend(rows)

In [34]:
df = pd.DataFrame(all_rows)
df.shape

(19501, 24)

In [35]:
print("API 전체 데이터 :", total_count)
print("실제 수집 데이터 :", len(df))
print("데이터 크기 :", df.shape)
print("중복 행 :", df.duplicated().sum())

API 전체 데이터 : 19501
실제 수집 데이터 : 19501
데이터 크기 : (19501, 24)
중복 행 : 0


# 데이터 전처리

## 불필요 컬럼 제거

In [36]:
df.head()

,CODENAME,GUNAME,TITLE,DATE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,INQUIRY,PLAYER,...,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,HMPG_ADDR,PRO_TIME
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",2026-12-24~2026-12-24,강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-23,시민,2026-12-24 00:00:00.0,2026-12-24 00:00:00.0,기타,127.157342546961,37.5512204558342,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",2026-12-22~2026-12-22,영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",070-8680-8477 / 02-337-3103,"Piano : Kazumi Tateishi, Contrabass : Shinobu ...",...,2026-07-16,시민,2026-12-22 00:00:00.0,2026-12-22 00:00:00.0,기타,126.900109255921,37.5260087284496,유료,https://culture.seoul.go.kr/culture/culture/cu...,19:30
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],2026-11-29~2026-11-29,마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),"02-3274-8600 [문의1번] 평일 9:00 ~ 18:00 (토,일 공휴일 휴무)",,...,2026-08-04,기관,2026-11-29 00:00:00.0,2026-11-29 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,(일) 16:00
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),2026-11-27~2026-11-29,동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,031-921-6325,,...,2026-07-21,시민,2026-11-27 00:00:00.0,2026-11-29 00:00:00.0,기타,127.00977973484339,37.56735731522952,무료,https://culture.seoul.go.kr/culture/culture/cu...,10:00 ~ 19:00
4,클래식,마포구,[마포문화재단] 제11회 M 클래식 축제 [뮤라벨 콘서트],2026-10-28~2026-10-28,마포아트센터 플레이맥,마포문화재단,8세이상 관람가능,"전석 20,000원","02-3274-8600 [문의1번] 평일 9:00 - 18:00 (토,일 공휴일 휴무)",,...,2026-07-08,기관,2026-10-28 00:00:00.0,2026-10-28 00:00:00.0,기타,126.945533810385,37.5499060881738,유료,https://culture.seoul.go.kr/culture/culture/cu...,(수) 19:30


In [37]:
df.columns

Index(['CODENAME', 'GUNAME', 'TITLE', 'DATE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'INQUIRY', 'PLAYER', 'PROGRAM', 'ETC_DESC', 'ORG_LINK',
       'MAIN_IMG', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'HMPG_ADDR', 'PRO_TIME'],
      dtype='str')

In [38]:
df = df[['CODENAME', 'GUNAME', 'TITLE', 'PLACE', 'ORG_NAME', 'USE_TRGT',
       'USE_FEE', 'RGSTDATE', 'TICKET', 'STRTDATE', 'END_DATE', 'THEMECODE',
       'LOT', 'LAT', 'IS_FREE', 'PRO_TIME', 'ORG_LINK', 'HMPG_ADDR']]

## 데이터 타입 변경

In [39]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19501 entries, 0 to 19500
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   CODENAME   19501 non-null  str  
 1   GUNAME     19501 non-null  str  
 2   TITLE      19501 non-null  str  
 3   PLACE      19501 non-null  str  
 4   ORG_NAME   19501 non-null  str  
 5   USE_TRGT   19501 non-null  str  
 6   USE_FEE    19501 non-null  str  
 7   RGSTDATE   19501 non-null  str  
 8   TICKET     19501 non-null  str  
 9   STRTDATE   19501 non-null  str  
 10  END_DATE   19501 non-null  str  
 11  THEMECODE  19501 non-null  str  
 12  LOT        19501 non-null  str  
 13  LAT        19501 non-null  str  
 14  IS_FREE    19501 non-null  str  
 15  PRO_TIME   19501 non-null  str  
 16  ORG_LINK   19501 non-null  str  
 17  HMPG_ADDR  19501 non-null  str  
dtypes: str(18)
memory usage: 2.7 MB


In [40]:
df['STRTDATE'] = pd.to_datetime(df['STRTDATE'])
df['END_DATE'] = pd.to_datetime(df['END_DATE'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19501 entries, 0 to 19500
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CODENAME   19501 non-null  str           
 1   GUNAME     19501 non-null  str           
 2   TITLE      19501 non-null  str           
 3   PLACE      19501 non-null  str           
 4   ORG_NAME   19501 non-null  str           
 5   USE_TRGT   19501 non-null  str           
 6   USE_FEE    19501 non-null  str           
 7   RGSTDATE   19501 non-null  str           
 8   TICKET     19501 non-null  str           
 9   STRTDATE   19501 non-null  datetime64[us]
 10  END_DATE   19501 non-null  datetime64[us]
 11  THEMECODE  19501 non-null  str           
 12  LOT        19501 non-null  str           
 13  LAT        19501 non-null  str           
 14  IS_FREE    19501 non-null  str           
 15  PRO_TIME   19501 non-null  str           
 16  ORG_LINK   19501 non-null  str           
 17  HMPG

In [41]:
df.head()

,CODENAME,GUNAME,TITLE,PLACE,ORG_NAME,USE_TRGT,USE_FEE,RGSTDATE,TICKET,STRTDATE,END_DATE,THEMECODE,LOT,LAT,IS_FREE,PRO_TIME,ORG_LINK,HMPG_ADDR
0,콘서트,강동구,"2026 카즈미 타테이시 트리오 내한공연-크리스마스, 재즈를 만나다-(서울)",강동아트센터 대극장 한강,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원 / A석 55...",2026-07-23,시민,2026-12-24,2026-12-24,기타,127.157342546961,37.5512204558342,유료,19:30,https://tickets.interpark.com/goods/26010350,https://culture.seoul.go.kr/culture/culture/cu...
1,콘서트,영등포구,"2026 카즈미 타테이시 트리오 내한공연-지브리, 재즈를 만나다-(서울)",영등포아트홀,기타,"성인, 청소년","VIP석 88,000원 / R석 77,000원 / S석 66,000원",2026-07-16,시민,2026-12-22,2026-12-22,기타,126.900109255921,37.5260087284496,유료,19:30,https://tickets.interpark.com/goods/26010060,https://culture.seoul.go.kr/culture/culture/cu...
2,연극,마포구,[마포문화재단] 체홉 4대 장막 낭독극 [공놀이클럽의 사계절 체홉: 갈매기],마포아트센터 아트홀맥,마포문화재단,14세 이상(2014년 이전 출생),전석 2만원(균일가),2026-08-04,기관,2026-11-29,2026-11-29,기타,126.945533810385,37.5499060881738,유료,(일) 16:00,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...
3,전시/미술,중구,파인캐릭터 2026 (FineCharacter 2026),동대문디자인플라자(DDP) 쇼룸 1층 (서울 중구 을지로 281),기타,누구나,,2026-07-21,시민,2026-11-27,2026-11-29,기타,127.00977973484339,37.56735731522952,무료,10:00 ~ 19:00,https://finecharacter.kr/,https://culture.seoul.go.kr/culture/culture/cu...
4,클래식,마포구,[마포문화재단] 제11회 M 클래식 축제 [뮤라벨 콘서트],마포아트센터 플레이맥,마포문화재단,8세이상 관람가능,"전석 20,000원",2026-07-08,기관,2026-10-28,2026-10-28,기타,126.945533810385,37.5499060881738,유료,(수) 19:30,https://www.mfac.or.kr/performance/whole_view....,https://culture.seoul.go.kr/culture/culture/cu...


## 결측값 체크

In [42]:
df.isna().sum()

CODENAME     0
GUNAME       0
TITLE        0
PLACE        0
ORG_NAME     0
USE_TRGT     0
USE_FEE      0
RGSTDATE     0
TICKET       0
STRTDATE     0
END_DATE     0
THEMECODE    0
LOT          0
LAT          0
IS_FREE      0
PRO_TIME     0
ORG_LINK     0
HMPG_ADDR    0
dtype: int64

In [43]:
len(df['USE_TRGT'].unique())

5006

In [44]:
df["USE_TRGT"].value_counts().head(30)

USE_TRGT
누구나                           4212
시민 누구나                         734
전체관람가                          677
성인                             481
홈페이지 참고                        469
초등학생 이상                        318
만 7세 이상                        304
8세 이상                          288
어린이                            166
초등학생 이상 관람가                    164
전체 관람가                         154
관심있는 누구나                       150
전체                             149
미취학아동 입장불가                     125
전 연령                           125
8세 이상 관람가                      107
36개월 이상                        104
프로그램별 상이                       103
7세 이상                           95
7세 이상 관람 가능 (2018년 이전 출생자)      89
모든 시민                           87
전연령                             82
5세 이상 어린이                       67
누구나                             66
7세 이상 관람 가능 (2019년 이전 출생자)      63
서울도서관 회원                        62
만 7세 이상                         61
일반시민                            59
초등학생       

In [48]:
df[df["USE_TRGT"] == "홈페이지 참고"]['HMPG_ADDR']

391      https://culture.seoul.go.kr/culture/culture/cu...
1071     https://culture.seoul.go.kr/culture/culture/cu...
2276     https://culture.seoul.go.kr/culture/culture/cu...
2485     https://culture.seoul.go.kr/culture/culture/cu...
3240     https://culture.seoul.go.kr/culture/culture/cu...
                               ...                        
18359    https://culture.seoul.go.kr/culture/culture/cu...
18370    https://culture.seoul.go.kr/culture/culture/cu...
18424    https://culture.seoul.go.kr/culture/culture/cu...
18542    https://culture.seoul.go.kr/culture/culture/cu...
18566    https://culture.seoul.go.kr/culture/culture/cu...
Name: HMPG_ADDR, Length: 469, dtype: str

행사 대상이 불명확한 데이터가 존재 함.


In [64]:
df["event_year"] = df["STRTDATE"].dt.year

df["event_year"].value_counts().sort_index()

event_year
2021     955
2022    3075
2023    3567
2024    5537
2025    3918
2026    2449
Name: count, dtype: int64

### 행사 시작일이 24년부터 현재까지의 데이터와 향후 예정인 데이터 대상으로 분석

In [65]:
df["STRTDATE"] = pd.to_datetime(
    df["STRTDATE"],
    errors="coerce"
)

start_date = pd.Timestamp("2024-01-01")

df_recent = df[
    df["STRTDATE"] >= start_date
].copy()

In [66]:
print("전체 행사 :", len(df))
print("최근 2년 행사 :", len(df_recent))

print(
    df_recent["STRTDATE"].min(),
    df_recent["STRTDATE"].max())

전체 행사 : 19501
최근 2년 행사 : 11904
2024-01-01 00:00:00 2026-12-24 00:00:00


In [68]:
df.to_csv(
    "../data/cultural_events_raw.csv",
    index=False,
    encoding="utf-8-sig"
)

In [67]:
df_recent.to_csv(
    "../data/cultural_events_2024_present.csv",
    index=False,
    encoding="utf-8-sig"
)